In [ ]:
# @title Install required dependencies
!pip install -q "crewai[tools]>=1.8.0" python-dotenv ddgs pydantic pandas pypdf


In [ ]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import FileWriterTool

SAMPLE_TRANSCRIPT = """
Alex: Let's finalize the Q3 launch date. I suggest August 15th.
Sarah: August 15th works if the backend migration finishes by July 28th.
David: I can finish the migration by July 28th, but I need two contractors approved by Friday.
Alex: Approved. Sarah, please coordinate with HR to get contractors onboarded by Monday.
Sarah: Will do. What about marketing collateral?
Alex: Elena is handling marketing; initial landing page review is set for August 5th.
David: Security audit is scheduled for August 8th. Any blockers there?
Alex: None known. Let's make sure test reports are published by August 10th.
"""

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = LLM(
        model="google/gemini-3.1-flash-lite-preview",
        api_key=userdata.get("GEMINI_API_KEY"),
        temperature=0.3,
        max_tokens=1200,
    )

    print("Using meeting transcript for synthesis...")

    summarizer = Agent(
        role="Meeting Summarizer",
        goal="Extract high-level discussion points and key takeaways from meeting transcripts",
        backstory="Executive assistant expert in synthesizing long discussions into crisp briefs.",
        llm=llm,
        verbose=True,
        allow_delegation=False,
    )

    action_tracker = Agent(
        role="Action Items Specialist",
        goal="Identify concrete action items, assignees, deadlines, and prerequisites",
        backstory="Project manager who ensures every task has clear accountability and due dates.",
        llm=llm,
        verbose=True,
        allow_delegation=False,
    )

    editor = Agent(
        role="Documentation Editor",
        goal="Format the executive summary and action items into a clean meeting brief",
        backstory="Technical communicator who prepares publication-ready team memos.",
        llm=llm,
        tools=[FileWriterTool()],
        verbose=True,
        allow_delegation=False,
    )

    summary_task = Task(
        description="Analyze the transcript: {transcript}. Summarize core objectives and key decisions reached.",
        expected_output="An executive summary outlining main topics discussed.",
        agent=summarizer,
    )

    action_task = Task(
        description="Extract all explicit and implicit action items with assigned owners and deadlines from: {transcript}.",
        expected_output="A table of action items with Owner, Task, and Deadline.",
        agent=action_tracker,
    )

    compile_task = Task(
        description="Combine executive summary and action items into a unified Markdown document. Save to output/meeting_notes.md.",
        expected_output="Complete meeting memo in Markdown.",
        agent=editor,
        context=[summary_task, action_task],
    )

    crew = Crew(
        agents=[summarizer, action_tracker, editor],
        tasks=[summary_task, action_task, compile_task],
        process=Process.sequential,
        verbose=True,
    )

    result = crew.kickoff(inputs={"transcript": SAMPLE_TRANSCRIPT})
    print("\n--- Meeting Notes Summary ---\n")
    print(result.raw)

except Exception as e:
    print("Error:", e)
